# Regression trees
The study-hours examples, the split search by squared error, and a regression tree on the Boston housing data.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

## Hours the day before the exam: a line fails, a tree works
The study-hours data are synthetic (fixed seeds), built to show the pattern.

In [2]:
day = pd.read_csv("data/exam_day.csv")
line = LinearRegression().fit(day[["hours"]], day["marks"])
print("line R2 on the training data:", round(line.score(day[["hours"]], day["marks"]), 3))
print("line predicts for 5 hours:", round(line.predict(pd.DataFrame({"hours": [5]}))[0], 1))

# A tree with at most 3 leaves finds the two cuts by itself
tree = DecisionTreeRegressor(max_leaf_nodes=3, random_state=0).fit(day[["hours"]], day["marks"])
print(export_text(tree, feature_names=["hours"]))
print("tree R2:", round(tree.score(day[["hours"]], day["marks"]), 3))

line R2 on the training data: 0.016
line predicts for 5 hours: 64.7
|--- hours <= 2.90
|   |--- value: [43.00]
|--- hours >  2.90
|   |--- hours <= 5.75
|   |   |--- value: [87.90]
|   |--- hours >  5.75
|   |   |--- value: [59.00]

tree R2: 0.964


In [3]:
# The hand-made tree: means of the three groups (hours < 3, 3 to 6, 6 and more)
for lo, hi in [(0, 3), (3, 6), (6, 99)]:
    group = day["marks"][(day["hours"] >= lo) & (day["hours"] < hi)]
    print(f"{lo} <= hours < {hi}: {len(group)} students, mean {group.mean():.1f}")

0 <= hours < 3: 7 students, mean 43.0
3 <= hours < 6: 10 students, mean 87.9
6 <= hours < 99: 13 students, mean 59.0


## Scoring one split: the sum of squared errors

In [4]:
hours = np.array([1, 2, 4, 5, 8])
marks = np.array([40, 46, 88, 90, 58])

def sse_of_split(t):
    # Each side predicts its own mean; add up the squared errors of both sides
    left, right = marks[hours <= t], marks[hours > t]
    return ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()

for t in [1.5, 3, 4.5, 6.5]:
    print(f"hours <= {t}: SSE = {sse_of_split(t):.1f}")

hours <= 1.5: SSE = 1443.0
hours <= 3: SSE = 660.7
hours <= 4.5: SSE = 1880.0
hours <= 6.5: SSE = 2136.0


In [5]:
# The same search over all 30 students: every midpoint between neighbouring hours
h, m = day["hours"].to_numpy(), day["marks"].to_numpy()
xs = np.unique(h)
cands = (xs[:-1] + xs[1:]) / 2
sse = [((m[h <= t] - m[h <= t].mean()) ** 2).sum() + ((m[h > t] - m[h > t].mean()) ** 2).sum() for t in cands]
print("best threshold:", cands[int(np.argmin(sse))], "SSE", round(min(sse), 1), "| SSE with no split:", round(((m - m.mean()) ** 2).sum(), 1))

best threshold: 2.9 SSE 5059.7 | SSE with no split: 9438.7


## Two inputs: hours and CGPA

In [6]:
two = pd.read_csv("data/exam_cgpa.csv")
t2 = DecisionTreeRegressor(max_depth=3, min_samples_leaf=4, random_state=0).fit(two[["hours", "cgpa"]], two["marks"])
print(export_text(t2, feature_names=["hours", "cgpa"]))

|--- hours <= 2.80
|   |--- cgpa <= 6.90
|   |   |--- cgpa <= 6.35
|   |   |   |--- value: [34.31]
|   |   |--- cgpa >  6.35
|   |   |   |--- value: [41.00]
|   |--- cgpa >  6.90
|   |   |--- cgpa <= 8.55
|   |   |   |--- value: [47.29]
|   |   |--- cgpa >  8.55
|   |   |   |--- value: [52.20]
|--- hours >  2.80
|   |--- hours <= 6.00
|   |   |--- cgpa <= 7.25
|   |   |   |--- value: [76.56]
|   |   |--- cgpa >  7.25
|   |   |   |--- value: [91.41]
|   |--- hours >  6.00
|   |   |--- cgpa <= 7.40
|   |   |   |--- value: [51.55]
|   |   |--- cgpa >  7.40
|   |   |   |--- value: [63.77]



## Regression tree on the Boston housing data

In [7]:
# load_boston was removed from scikit-learn in version 1.2; data/boston.csv is the same table (OpenML id 531)
boston = pd.read_csv("data/boston.csv")
X, y = boston.drop(columns="MEDV"), boston["MEDV"]       # MEDV: median house value, in thousands of dollars
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = DecisionTreeRegressor(criterion="squared_error", max_depth=5, random_state=42)
reg.fit(X_train, y_train)
print("test R2:", round(r2_score(y_test, reg.predict(X_test)), 3))
print("5-fold CV R2 on the training set:", round(cross_val_score(reg, X_train, y_train, cv=5).mean(), 3))

test R2: 0.883
5-fold CV R2 on the training set: 0.658


In [8]:
# Tune several hyperparameters at once with cross-validation
params = {"max_depth": [2, 4, 6, 8, None],
          "criterion": ["squared_error", "absolute_error"],
          "max_features": [0.25, 0.5, 1.0],
          "min_samples_split": [2, 0.05, 0.25]}
grid = GridSearchCV(DecisionTreeRegressor(random_state=42), params, cv=5)
grid.fit(X_train, y_train)
print(len(grid.cv_results_["params"]), "settings tried")
print("best:", grid.best_params_, "CV R2", round(grid.best_score_, 3), "test R2", round(grid.score(X_test, y_test), 3))

90 settings tried
best: {'criterion': 'absolute_error', 'max_depth': 6, 'max_features': 1.0, 'min_samples_split': 2} CV R2 0.725 test R2 0.632


### How optimistic is the grid's best score?
The grid's 0.725 is the best of 90 scores measured on the same 5 folds. We re-score the tuned setting on 10 fresh shuffles of 5-fold CV (only the folds change).

In [9]:
from sklearn.model_selection import RepeatedKFold
rcv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=0)
tuned = DecisionTreeRegressor(random_state=42, **grid.best_params_)
print("grid's best CV R2:", round(grid.best_score_, 3))
print("same setting, repeated 5-fold CV R2 on fresh folds:", round(cross_val_score(tuned, X_train, y_train, cv=rcv).mean(), 3))

grid's best CV R2: 0.725


same setting, repeated 5-fold CV R2 on fresh folds: 0.663


## Does tuning beat a fully grown tree?
A tree with default settings grows until its leaves are pure, so it fits the noise in the target. We test this on the California housing data (20,640 districts, 8 features, a noisy target). For each of 20 random train-test splits, we tune `max_depth` and `min_samples_leaf` with 5-fold CV **on the training part only**, then score the tuned tree and an untuned tree on the test part, which the search never saw.

In [10]:
from sklearn.datasets import fetch_california_housing
Xc, yc = fetch_california_housing(return_X_y=True, as_frame=True)   # target: median house value, in 100,000s of dollars
small_grid = {"max_depth": [4, 6, 8, 10, 12, None], "min_samples_leaf": [1, 5, 20, 50]}
rows = []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.2, random_state=seed)
    full = DecisionTreeRegressor(random_state=42).fit(Xa, ya)          # untuned: grown until the leaves are pure
    search = GridSearchCV(DecisionTreeRegressor(random_state=42), small_grid, cv=5, n_jobs=-1).fit(Xa, ya)
    rows.append({"split": seed, "full_train": full.score(Xa, ya), "full_test": full.score(Xb, yb),
                 "tuned_test": search.score(Xb, yb), "leaves_full": full.get_n_leaves(),
                 "leaves_tuned": search.best_estimator_.get_n_leaves(), **search.best_params_})
res = pd.DataFrame(rows)
res.to_csv("data/california_tuning.csv", index=False)
print(res[["full_train", "full_test", "tuned_test", "leaves_full", "leaves_tuned"]].mean().round(3))
print("test R2 std: full", round(res.full_test.std(), 3), "| tuned", round(res.tuned_test.std(), 3))
print("tuned beats full on", (res.tuned_test > res.full_test).sum(), "of", len(res), "splits")
print(res[["max_depth", "min_samples_leaf"]].value_counts())

full_train          1.000
full_test           0.612
tuned_test          0.730
leaves_full     15856.600
leaves_tuned      631.950
dtype: float64
test R2 std: full 0.024 | tuned 0.014
tuned beats full on 20 of 20 splits
max_depth  min_samples_leaf
12.0       20                  1
Name: count, dtype: int64


In [11]:
# RandomizedSearchCV tries only n_iter random settings from the same grid: faster on big grids
rand = RandomizedSearchCV(DecisionTreeRegressor(random_state=42), params, n_iter=20, cv=5, random_state=42)
rand.fit(X_train, y_train)
print("best:", rand.best_params_, "CV R2", round(rand.best_score_, 3))

best: {'min_samples_split': 2, 'max_features': 0.25, 'max_depth': None, 'criterion': 'absolute_error'} CV R2 0.704


In [12]:
# Feature importance: each column's share of the total impurity reduction in the tree (here: absolute error)
imp = pd.Series(grid.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(3))
print("sum:", imp.sum())

RM         0.469
LSTAT      0.291
CRIM       0.106
DIS        0.028
AGE        0.025
NOX        0.023
B          0.020
PTRATIO    0.019
TAX        0.013
INDUS      0.006
ZN         0.000
CHAS       0.000
RAD        0.000
dtype: float64
sum: 1.0


In [13]:
# absolute_error against squared_error: training time of one fully grown tree on all 506 rows
import time
for crit in ["squared_error", "absolute_error"]:
    DecisionTreeRegressor(criterion=crit, random_state=42).fit(X, y)          # warm-up
    t0 = time.perf_counter()
    for _ in range(20):
        DecisionTreeRegressor(criterion=crit, random_state=42).fit(X, y)
    print(crit, "seconds per fit:", round((time.perf_counter() - t0) / 20, 4))

squared_error seconds per fit: 0.0143


absolute_error seconds per fit: 0.0317


In [14]:
# How stable are the importances? Refit on 30 bootstrap samples: one tree against a random forest
from sklearn.ensemble import RandomForestRegressor
rng = np.random.default_rng(0)
tree_imp, forest_imp = [], []
for s in range(30):
    idx = rng.integers(0, len(X), len(X))
    tree_imp.append(DecisionTreeRegressor(random_state=s).fit(X.iloc[idx], y.iloc[idx]).feature_importances_)
    forest_imp.append(RandomForestRegressor(n_estimators=100, random_state=s, n_jobs=-1).fit(X.iloc[idx], y.iloc[idx]).feature_importances_)
tree_imp, forest_imp = pd.DataFrame(tree_imp, columns=X.columns), pd.DataFrame(forest_imp, columns=X.columns)
for col in ["RM", "LSTAT", "CRIM"]:
    print(col, "tree std", round(tree_imp[col].std(), 3), "| forest std", round(forest_imp[col].std(), 3))

RM tree std 0.203 | forest std 0.123
LSTAT tree std 0.203 | forest std 0.116
CRIM tree std 0.022 | forest std 0.011


In [15]:
# The criterion values DecisionTreeRegressor accepts in this version
import sklearn
print(sklearn.__version__, [c.options for c in DecisionTreeRegressor._parameter_constraints["criterion"] if hasattr(c, "options")])

1.9.1 [{'absolute_error', 'squared_error', 'poisson'}]
